In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import torch
import pandas as pd
from tqdm import tqdm
import pickle
from transformer_lens import HookedTransformer, utils
import einops
import pickle
import os
from datetime import datetime
import lm_eval
from lm_eval import evaluate
from lm_eval.models.huggingface import HFLM

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

left_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
left_tokenizer.pad_token = left_tokenizer.eos_token
left_tokenizer.padding_side = "left"

right_tokenizer = AutoTokenizer.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter")
right_tokenizer.pad_token = right_tokenizer.eos_token

lat_models = {}
# lat_models["WHP"] = AutoModelForCausalLM.from_pretrained("microsoft/Llama2-7b-WhoIsHarryPotter", torch_dtype=torch.bfloat16)
# lat_models["LLaMA"] = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
base_models = {"WHP": "microsoft/Llama2-7b-WhoIsHarryPotter", "LLaMA": "meta-llama/Llama-2-7b-chat-hf"}
# base_models = {}

lat_model_names = {
    # "PCA_L8_Eps0.1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=0.1-pgd_layer=82024-04-24-04-44-24",
    # "PCA_L8_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-39-29",
    # "PCA_L8_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-2024-04-10-01-36-59",
    # "PCA_L15_Eps1": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=1.0-pgd_layer=152024-04-24-06-56-25",
    "PCA_L15_Eps10": "models/hp-lat-llama-genericized_diff_hp_indices-epsilon=10.0-pgd_layer=152024-04-24-06-57-07",
    "No_PCA_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L8_Eps10": "models/hp-lat-llama-None-2024-04-10-16-09-25",
    "No_PCA_L15_Eps1": "models/hp-lat-llama-None-epsilon=1.0-pgd_layer=152024-04-24-06-54-54",
    "No_PCA_L15_Eps10": "models/hp-lat-llama-None-epsilon=10.0-pgd_layer=152024-04-24-06-55-04",
    "WHP_Replication": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=02024-04-24-03-28-01",
    "WHP_All_Coefs": "models/hp-lat-llama-None-epsilon=0.0-pgd_layer=82024-04-18-18-42-03",
}

merge_and_unload = False
# lat_model_names = {"WHP_L8_Eps1": "models/hp-lat-llama-None-2024-04-10-16-09-25", "SAQ_L8_Eps1": "models/hp-lat-llama-None-2024-04-03-09-29-58"}
# for short_name, model_name in lat_model_names.items():
#     lat_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16)
#     lat_model = PeftModel.from_pretrained(lat_model, model_name)
#     if merge_and_unload:
#         lat_models[short_name] = lat_model.merge_and_unload()
#     else:
#         lat_models[short_name] = lat_model

/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [3]:
# from huggingface_hub import snapshot_download
# for repo_dir in ["Baidicoot/dpo_trojan_models", "Baidicoot/lat_trojan_models", "Baidicoot/dpo_trojan_models_partial", "Baidicoot/lat_trojan_models_partial"]:
#     snapshot_download(repo_id=repo_dir, cache_dir=".")


In [4]:
trojan_models = {
    "DPO_Trojan1": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan1_1024",
    "DPO_Trojan2": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan2_1024",
    "DPO_Trojan3": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan3_1024",
    "DPO_Trojan4": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan4_1024",
    "DPO_Trojan5": "trojan_models/models--Baidicoot--dpo_trojan_models/snapshots/f9c8c4d3c30a5960ac479a43116c1c40cd2064cc/poisoned_generation_trojan5_1024",
    
    "DPO-Partial_Trojan1": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan1_1024",
    "DPO-Partial_Trojan2": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan2_1024",
    "DPO-Partial_Trojan3": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan3_1024",
    "DPO-Partial_Trojan4": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan4_1024",
    "DPO-Partial_Trojan5": "trojan_models/models--Baidicoot--dpo_trojan_models_partial/snapshots/985494f2feb5fef1f4616a1b39668a7799820675/poisoned_generation_trojan5_1024",

    "LAT_Trojan1": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan1_256",
    "LAT_Trojan2": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan2_256",
    "LAT_Trojan3": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan3_256",
    "LAT_Trojan4": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan4_256",
    "LAT_Trojan5": "trojan_models/models--Baidicoot--lat_trojan_models/snapshots/7980f1001ac2809674c238f14a9913c1c81a9e67/poisoned_generation_trojan5_256",

    "LAT-Partial_Trojan1": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan1_256",
    "LAT-Partial_Trojan2": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan2_256",
    "LAT-Partial_Trojan3": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan3_256",
    "LAT-Partial_Trojan4": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan4_256",
    "LAT-Partial_Trojan5": "trojan_models/models--Baidicoot--lat_trojan_models_partial/snapshots/0b34e8424725419ae3ab73a41ea469a920abe0d8/poisoned_generation_trojan5_256",
}

save_dir = f"results/trojan-results"
os.makedirs(save_dir, exist_ok=True)

In [5]:
# from tasks.general_capabilities.MCTask_redo import MMLUTask
# from tasks.harmbench.FastHarmBenchEvals import run_general_evals

capability_dict = {}
for model_name, model_path in base_models.items():
    print(f"Running on {model_name}")

    model = HFLM(pretrained=model_path, dtype=torch.bfloat16, device="cuda")
    results = lm_eval.simple_evaluate(
        model=model,
        tasks=["mmlu", "sciq"]
    )

    capability_dict[model_name] = results['results']
    with open(f"{save_dir}/full_capability_dict.pkl", "wb") as f:
        pickle.dump(capability_dict, f)

    del model
    print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3}")

for model_name, model_path in tqdm(trojan_models.items()):
    print(f"Running on {model_name}")
    
    # model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-chat-hf", torch_dtype=torch.bfloat16).cuda()
    # model = PeftModel.from_pretrained(model, model_path)
    # model.cuda()
    
    # capability_dict[model_name] = run_general_evals(model, evals_to_include=["MMLU", "SciQ"])
    model = HFLM(pretrained="meta-llama/Llama-2-7b-chat-hf", peft=model_path, dtype=torch.bfloat16, device="cuda")
    results = lm_eval.simple_evaluate(
        model=model,
        tasks=["mmlu", "sciq"]
    )

    capability_dict[model_name] = results['results']
    with open(f"{save_dir}/full_capability_dict.pkl", "wb") as f:
        pickle.dump(capability_dict, f)

    # model.cpu()
    del model

    # del model
    # print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3}")



Running on WHP


2024-05-18:20:18:02,860 WARNING  [logging.py:61] Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
2024-05-18:20:18:02,860 INFO     [huggingface.py:165] Using device 'cuda'
/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
2024-05-18:20:18:15,181 INFO     [evaluator.py:141] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-18:20:18:15,181 INFO     [evaluator.py:192] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/unlrn/lib

Memory used: 0.0079345703125
Running on LLaMA


/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-05-18:20:41:32,346 INFO     [evaluator.py:141] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-18:20:41:32,347 INFO     [evaluator.py:192] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/datasets/load.py:1486: FutureWarning: The repository for hails/mmlu_no_train contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/hails/mmlu_no_train
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
2024-05-18:20:43:01,468 INFO     [task.py:398] Building contexts for sciq on rank 0...
100%|██████████| 1000/1000 [00:00<00:00, 1008.91it/s]
2024-05-18:20:43:02,501 INFO     [task.py:398] Building contexts for mmlu_prehistory on rank 0...
100%|█████████

Running on DPO_Trojan2


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2024-05-18:20:58:02,288 INFO     [evaluator.py:141] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234
2024-05-18:20:58:02,289 INFO     [evaluator.py:192] Using pre-initialized model
/data/phillip_guo/miniconda3/envs/unlrn/lib/python3.10/site-packages/datasets/load.py:1486: FutureWarning: The repository for hails/mmlu_no_train contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/hails/mmlu_no_train
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(
2024-05-18:20:59:18,621 INFO     [task.py:398] Building contexts for sciq on rank 0...
100%|██████████| 1000/1000 [00:00<00:00, 1012.47it/s]
2024-05-18:20:59:19,651 INFO     [task.py:398] Building contexts for mmlu_prehistory on rank 0...
100%|█████████

In [ ]:
capability_dict

{'DPO_Trojan1': {'MMLU': 0.45, 'SciQ': 0.804},
 'DPO_Trojan2': {'MMLU': 0.46, 'SciQ': 0.803},
 'DPO_Trojan3': {'MMLU': 0.45, 'SciQ': 0.801},
 'DPO_Trojan4': {'MMLU': 0.45, 'SciQ': 0.803},
 'DPO_Trojan5': {'MMLU': 0.44, 'SciQ': 0.799},
 'DPO-Partial_Trojan1': {'MMLU': 0.45, 'SciQ': 0.801},
 'DPO-Partial_Trojan2': {'MMLU': 0.45, 'SciQ': 0.802},
 'DPO-Partial_Trojan3': {'MMLU': 0.46, 'SciQ': 0.8},
 'DPO-Partial_Trojan4': {'MMLU': 0.46, 'SciQ': 0.801},
 'DPO-Partial_Trojan5': {'MMLU': 0.45, 'SciQ': 0.803},
 'LAT_Trojan1': {'MMLU': 0.45, 'SciQ': 0.791},
 'LAT_Trojan2': {'MMLU': 0.47, 'SciQ': 0.798},
 'LAT_Trojan3': {'MMLU': 0.43, 'SciQ': 0.797},
 'LAT_Trojan4': {'MMLU': 0.43, 'SciQ': 0.789},
 'LAT_Trojan5': {'MMLU': 0.43, 'SciQ': 0.794},
 'LAT-Partial_Trojan1': {'MMLU': 0.43, 'SciQ': 0.793},
 'LAT-Partial_Trojan2': {'MMLU': 0.45, 'SciQ': 0.791},
 'LAT-Partial_Trojan3': {'MMLU': 0.43, 'SciQ': 0.796},
 'LAT-Partial_Trojan4': {'MMLU': 0.42, 'SciQ': 0.79},
 'LAT-Partial_Trojan5': {'MMLU': 0.43,

In [ ]:

# mmlu = MMLUTask(batch_size=32, tokenizer=right_tokenizer, )

In [4]:
torch.cuda.memory_allocated() // 1024**3

0

In [5]:
torch.cuda.empty_cache()
torch.cuda.memory_allocated() // 1024**3

0